In [3]:
from data_gen import make_deliveries
df , late = make_deliveries()
df.columns

Index(['distance', 'prep', 'hour', 'weather', 'vehicle', 'is_rush'], dtype='object')

In [4]:
late

array([1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1,
       1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0,
       0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0,
       0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0,
       0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0,
       0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0,
       1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1,
       0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1,

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

df , late = make_deliveries()
df.drop(columns  = 'is_rush',inplace=True)

categorical = df.select_dtypes(include=['object','category']).columns
numerical = df.select_dtypes(exclude=['object','category']).columns



In [24]:
pre = ColumnTransformer([
    ("num", StandardScaler(), numerical),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])

models = {
    "Logistic Regression": Pipeline([
        ("prep", pre),
        ("clf", LogisticRegression(max_iter=1000))
    ]),

    "Decision Tree (max_depth=4)": Pipeline([
        ("prep", pre),
        ("clf", DecisionTreeClassifier(max_depth=4))
    ]),

    "Random Forest (300 trees)": Pipeline([
        ("prep", pre),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            random_state=42
        ))
    ]),
}

In [25]:
for name, m in models.items():
    scores = cross_val_score(m, df, late, cv=5)
    print(f"{name}: {round(scores.mean(), 3)}, {round(scores.std(), 3)}")

Logistic Regression: 0.758, 0.022
Decision Tree (max_depth=4): 0.709, 0.024
Random Forest (300 trees): 0.765, 0.031
